# 02 — Train DDPM or Flow Matching on Kolmogorov Flow

Set `MODEL_TYPE` in the config cell to either `'ddpm'` or `'fm'` and run all.

Run this twice: once for each model. Checkpoints go to `checkpoints/{model_type}_{run_name}/`.

In [ ]:
# --- Bootstrap ---
import sys
from pathlib import Path

try:
    import google.colab  # noqa
    IN_COLAB = True
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    REPO = Path('/content/drive/MyDrive/courses/24788-Intro_of_DL/project/code')
except ImportError:
    IN_COLAB = False
    REPO = Path.cwd()
    while not (REPO / 'requirements.txt').exists() and REPO != REPO.parent:
        REPO = REPO.parent

assert (REPO / 'src' / 'train.py').exists(), f'repo not found at {REPO}'
sys.path.insert(0, str(REPO))
print(f'IN_COLAB = {IN_COLAB}\nREPO     = {REPO}')

In [ ]:
# --- Install deps if needed (only in Colab) ---
if IN_COLAB:
    !pip install -q einops h5py

In [ ]:
# --- GPU check ---
import torch
print(f'torch   : {torch.__version__}')
print(f'cuda    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'device  : {torch.cuda.get_device_name(0)}')
    print(f'memory  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# =============================================================================
# CONFIG — change MODEL_TYPE and re-run for the other model
# =============================================================================
from src.train import TrainConfig, train

MODEL_TYPE = 'ddpm'   # 'ddpm' or 'fm'

cfg = TrainConfig(
    model_type=MODEL_TYPE,
    run_name='main',
    # Model
    base_ch=64,
    ch_mults=(1, 2, 2, 4),
    n_res_blocks=2,
    dropout=0.1,
    # Data
    batch_size=32,
    num_workers=2,
    # Optimization
    lr=2e-4,
    max_steps=40_000,
    warmup_steps=500,
    ema_decay=0.999,
    # Logging
    log_every=50,
    sample_every=2_000,
    ckpt_every=2_000,
    amp=True,
)
print(cfg)

In [ ]:
# --- Train (resumes automatically from the latest checkpoint) ---
run_dir = train(cfg)
print(f'\nDone. run_dir = {run_dir}')

In [ ]:
# --- Plot training loss ---
import json, matplotlib.pyplot as plt
state = torch.load(run_dir / 'state.pt', map_location='cpu', weights_only=False)
loss_log = state['loss_log']
steps = [x['step'] for x in loss_log]
losses = [x['loss'] for x in loss_log]
plt.figure(figsize=(7, 4))
plt.plot(steps, losses)
plt.xlabel('step'); plt.ylabel('loss')
plt.yscale('log')
plt.title(f'{MODEL_TYPE} training loss')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(REPO / 'results' / f'loss_curve_{MODEL_TYPE}.png', dpi=120)
plt.show()

---
**Next**: change `MODEL_TYPE = 'fm'` and re-run cells above to train the variant.

Once both are trained, move to `03_evaluate.ipynb`.